In [ ]:
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
# Load training data
url = "https://huggingface.co/datasets/usaaio-official/2026_USAAIO_Round1_public/raw/main/2026_USAAIO_Round1_breast_cancer_train.csv"
df = pd.read_csv(url)
X = df.drop("target", axis=1)
y = df["target"].values

In [ ]:
# Manual standardization - store params for inference
mu, sigma = X.mean().values, X.std().values
X_scaled = (X.values - mu) / sigma

In [ ]:
# Manual 5-fold CV to find best k (optimizing macro F1)
def f1_macro(y_true, y_pred):
    f1s = []
    for c in [0, 1]:
        tp = ((y_pred == c) & (y_true == c)).sum()
        fp = ((y_pred == c) & (y_true != c)).sum()
        fn = ((y_pred != c) & (y_true == c)).sum()
        prec = tp / (tp + fp) if tp + fp > 0 else 0
        rec = tp / (tp + fn) if tp + fn > 0 else 0
        f1s.append(2 * prec * rec / (prec + rec) if prec + rec > 0 else 0)
    return np.mean(f1s)

n = len(y)
idx = np.arange(n)
np.random.seed(42)
np.random.shuffle(idx)
folds = np.array_split(idx, 5)

best_k, best_score = 1, 0
for k in range(1, 31):
    scores = []
    for i in range(5):
        val_idx = folds[i]
        train_idx = np.concatenate([folds[j] for j in range(5) if j != i])
        knn = KNeighborsClassifier(n_neighbors=k)
        knn.fit(X_scaled[train_idx], y[train_idx])
        scores.append(f1_macro(y[val_idx], knn.predict(X_scaled[val_idx])))
    if np.mean(scores) > best_score:
        best_k, best_score = k, np.mean(scores)
print(f"Best k={best_k}, CV F1-macro={best_score:.4f}")

In [ ]:
# Train final model
model = KNeighborsClassifier(n_neighbors=best_k)
model.fit(X_scaled, y)

In [ ]:
def my_prediction(X_test):
    X_test_scaled = (X_test.values - mu) / sigma
    return pd.Series(model.predict(X_test_scaled))

## Summary

**Approach:** kNN with manual standardization and 5-fold CV for k selection.

**Design choices:**
- **Manual standardization:** (X - mean) / std, essential for distance-based kNN.
- **k selection:** Searched k=1-30 via 5-fold CV optimizing macro F1.
- **Minimal imports:** Only numpy, pandas, and KNeighborsClassifier.

**Alternatives considered:**
- Different distance metrics - Euclidean works well with standardized features.
- Feature selection/PCA - not needed for 30 features; keeps solution simple.